# 快速入门

本节将介绍机器学习中常见任务的 API。请参阅各节中的链接以深入了解。

## 处理数据

PyTorch 提供了两个用于处理数据的基本方法： torch.utils.data.DataLoader和torch.utils.data.Dataset。 Dataset存储样本及其对应的标签，并将DataLoader一个可迭代对象包装在Dataset。

In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

PyTorch 提供了一些特定领域的库，例如TorchText、 TorchVision和TorchAudio，它们都包含数据集。在本教程中，我们将使用 TorchVision 数据集。
该torchvision.datasets模块包含Dataset适用于多种真实世界视觉数据集的对象，例如 CIFAR 和 COCO（完整列表请点击此处）。在本教程中，我们将使用 FashionMNIST 数据集。每个 TorchVision 对象Dataset都包含两个参数： `samples` 和 `labels` transform，分别 target_transform用于修改样本和标签。

In [3]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

100%|█████████████████████████████████████████████████████████████████████████████| 26.4M/26.4M [00:09<00:00, 2.66MB/s]


Extracting data\FashionMNIST\raw\train-images-idx3-ubyte.gz to data\FashionMNIST\raw



100%|██████████████████████████████████████████████████████████████████████████████| 29.5k/29.5k [00:00<00:00, 125kB/s]


Extracting data\FashionMNIST\raw\train-labels-idx1-ubyte.gz to data\FashionMNIST\raw



100%|█████████████████████████████████████████████████████████████████████████████| 4.42M/4.42M [00:01<00:00, 2.52MB/s]


Extracting data\FashionMNIST\raw\t10k-images-idx3-ubyte.gz to data\FashionMNIST\raw



100%|█████████████████████████████████████████████████████████████████████████████████████| 5.15k/5.15k [00:00<?, ?B/s]

Extracting data\FashionMNIST\raw\t10k-labels-idx1-ubyte.gz to data\FashionMNIST\raw



我们将数据加载器Dataset作为参数传递给它DataLoader。这会封装一个包含我们数据集的可迭代对象，并支持自动批处理、采样、打乱顺序和多进程数据加载。这里我们定义批处理大小为 64，即数据加载器可迭代对象中的每个元素都会返回一个包含 64 个特征和标签的批次。

In [4]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


## 创建模型

要在 PyTorch 中定义神经网络，我们创建一个继承自`nn.Module`的类。我们在函数中定义网络的层__init__，并在函数中指定数据在网络中的传递方式forward。为了加速神经网络的运算，我们将其迁移到加速器，例如 CUDA、MPS、MTIA 或 XPU。如果当前有可用的加速器，我们将使用它；否则，我们将使用 CPU。

In [8]:
device = "cpu"
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

Using cpu device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


## 优化模型参数

要训练一个模型，我们需要一个损失函数 和一个优化器。

In [9]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

在一个训练循环中，模型对训练数据集（分批输入）进行预测，并将预测误差反向传播以调整模型的参数。

In [10]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

我们还会检查模型在测试数据集上的性能，以确保它能够学习。

In [11]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

训练过程需要经过多次迭代（epoch）。在每个epoch中，模型都会学习参数以做出更准确的预测。我们会记录每个epoch的模型准确率和损失值；我们希望看到准确率随着epoch的增加而提高，损失值随着epoch的增加而降低。

In [12]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.299903  [   64/60000]
loss: 2.290016  [ 6464/60000]
loss: 2.266941  [12864/60000]
loss: 2.260089  [19264/60000]
loss: 2.242223  [25664/60000]
loss: 2.212885  [32064/60000]
loss: 2.215135  [38464/60000]
loss: 2.180113  [44864/60000]
loss: 2.182729  [51264/60000]
loss: 2.144346  [57664/60000]
Test Error: 
 Accuracy: 44.2%, Avg loss: 2.142391 

Epoch 2
-------------------------------
loss: 2.161245  [   64/60000]
loss: 2.146218  [ 6464/60000]
loss: 2.081817  [12864/60000]
loss: 2.094159  [19264/60000]
loss: 2.041967  [25664/60000]
loss: 1.986578  [32064/60000]
loss: 2.006888  [38464/60000]
loss: 1.925778  [44864/60000]
loss: 1.940896  [51264/60000]
loss: 1.860750  [57664/60000]
Test Error: 
 Accuracy: 48.9%, Avg loss: 1.858720 

Epoch 3
-------------------------------
loss: 1.907212  [   64/60000]
loss: 1.870148  [ 6464/60000]
loss: 1.742738  [12864/60000]
loss: 1.783519  [19264/60000]
loss: 1.680124  [25664/60000]
loss: 1.633475  [32064/600

## 节省模型

保存模型的一种常见方法是序列化内部状态字典（包含模型参数）。

In [13]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


## 正在加载模型

加载模型的过程包括重新创建模型结构并将状态字典加载到其中.

In [14]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>

In [15]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Ankle boot", Actual: "Ankle boot"
